# AI Job Analyzer

**Kurs:** Utveckling med Python – grundnivå  
**Roll:** AI Engineer  
**Författare:** Carl Rhedin  

## Syfte

Detta projekt analyserar jobbannonser inom AI/IT och identifierar vilka tekniska kompetenser som efterfrågas. Programmet kan:

- Läsa jobbannonser från en lokal JSON-fil
- Hämta AI-jobb från RemoteOK:s publika API
- Matcha användarens kompetenser mot annonserna och beräkna matchningsprocent

## Koppling till AI-utvecklarrollen

AI-utvecklare arbetar ofta med att strukturera och analysera ostrukturerad text. Detta projekt demonstrerar:

- Hantering av extern data via API
- Textanalys och filtrering
- Objektorienterad programmering
- Felhantering och robust kod

In [1]:
# Standardbibliotek
import json
import re
import sys
from pathlib import Path

# Externa bibliotek
import ftfy
import requests

# Se till att kunna importera från src/
sys.path.append(str(Path.cwd().parent / "src"))

print("Imports klara.")

Imports klara.


## 1. Klasser (models.py)

Programmet använder tre klasser:

- **`JobAd`** – basklass för en jobbannons med titel och beskrivning.
- **`AIJob`** – ärver från `JobAd` och lägger till AI-område. Detta demonstrerar **arv**.
- **`JobAnalyzer`** – analyserar jobb mot en lista av kompetenser och beräknar matchning.

In [2]:
class JobAd:
    """En jobbannons."""

    def __init__(self, title, description):
        self.title = title
        self.description = description

    def has_skill(self, skill):
        """Returnerar True om kompetensen finns i annonsen."""
        text = f"{self.title} {self.description}"
        return skill.lower() in text.lower()

    def __str__(self):
        return self.title


class AIJob(JobAd):
    """En AI-specifik jobbannons. Ärver från JobAd."""

    def __init__(self, title, description, ai_area):
        super().__init__(title, description)
        self.ai_area = ai_area

    def __str__(self):
        return f"[AI] {super().__str__()}"


class JobAnalyzer:
    """Analyserar jobbannonser mot en lista av kompetenser."""

    def __init__(self, skills):
        self.skills = skills

    def analyze(self, job):
        """Returnerar en lista av kompetenser som finns i annonsen."""
        return [s for s in self.skills if job.has_skill(s)]

    def match(self, job, user_skills):
        """Returnerar matchningsresultat mellan användarens skills och annonsen."""
        required = set(s.lower() for s in self.analyze(job))
        if not required:
            return {"percent": None, "matching": [], "missing": []}

        user = set(s.lower() for s in user_skills)
        matching = required & user
        missing = required - user
        percent = round(len(matching) / len(required) * 100, 1)
        return {"percent": percent, "matching": sorted(matching), "missing": sorted(missing)}

In [3]:
# Snabbtest av klasserna
demo = AIJob("AI Engineer", "Vi söker Python-utvecklare.", "NLP")
print(demo)
print("Har Python:", demo.has_skill("Python"))
print("Har Java:", demo.has_skill("Java"))

analyzer = JobAnalyzer(["Python", "SQL", "Docker"])
print("Hittade:", analyzer.analyze(demo))
print("Matchning:", analyzer.match(demo, ["python", "sql"]))

[AI] AI Engineer
Har Python: True
Har Java: False
Hittade: ['Python']
Matchning: {'percent': 100.0, 'matching': ['python'], 'missing': []}


## 2. API-hämtning (api_client.py)

Funktionen `fetch_jobs()` hämtar AI-jobb från [RemoteOK:s publika API](https://remoteok.com/api).

- Använder `requests` för HTTP-anrop.
- Använder `ftfy` för att fixa dubbelkodad text från API:et.
- Använder `re` för att ta bort HTML-taggar och matcha nyckelord.
- Felhantering med `try/except` fångar nätverksfel.

In [5]:
API_URL = "https://remoteok.com/api"


def clean_text(text):
    """Tar bort HTML och fixar teckenkodning."""
    text = ftfy.fix_text(text)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def fetch_jobs(limit=5):
    """Hämtar AI-jobb från RemoteOK. Returnerar en lista av dicts."""
    headers = {"User-Agent": "ai-job-analyzer/1.0"}

    try:
        response = requests.get(API_URL, headers=headers, timeout=10)
        response.raise_for_status()
        data = json.loads(response.content.decode("utf-8", errors="replace"))
    except requests.exceptions.RequestException as e:
        print(f"Nätverksfel: {e}")
        return []

    keywords = ["ai", "machine learning", "ml", "deep learning", "nlp"]
    ai_jobs = []
    for job in data[1:]:
        title = job.get("position", "").lower()
        if any(re.search(r"\b" + re.escape(k) + r"\b", title) for k in keywords):
            ai_jobs.append(job)
        if len(ai_jobs) >= limit:
            break

    for job in ai_jobs:
        job["description"] = clean_text(job.get("description", ""))

    return ai_jobs

In [6]:
# Testa API
raw = fetch_jobs(limit=3)
print(f"Hämtade {len(raw)} jobb:")
for j in raw:
    print(f"- {j['position']}")

Hämtade 3 jobb:
- Technical Product Manager AI Stockbroking App
- Senior Growth Product Manager AI Native
- AI agent engineer


## 3. Analys och matchning

Nu kombinerar vi klasserna med API:et:

1. Läs lokala annonser från `data/jobads.json`
2. Hämta AI-jobb från API:et
3. Analysera och matcha mot användarens kompetenser

**Matchningsformeln:**

Om annonsen inte innehåller några av våra sökta kompetenser, returneras `None` (visas som `–`) istället för `0%` – för att skilja på "ingen data" och "ingen matchning".


In [7]:
SKILLS = ["Python", "SQL", "Git", "Docker", "AWS",
          "TensorFlow", "PyTorch", "Machine Learning", "NLP"]


def load_local_jobs(path="data/jobads.json"):
    """Läser lokala annonser från JSON."""
    try:
        with open(Path.cwd().parent / path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        return [JobAd(ad["title"], ad["description"]) for ad in raw]
    except (FileNotFoundError, json.JSONDecodeError, KeyError) as e:
        print(f"Kunde inte läsa {path}: {e}")
        return []


def load_api_jobs():
    """Hämtar AI-jobb från API."""
    raw = fetch_jobs(limit=5)
    return [AIJob(j["position"], j.get("description", ""), "AI") for j in raw]


def analyze_jobs(jobs, analyzer):
    """Analyserar jobb och skriver ut resultat."""
    for job in jobs:
        found = analyzer.analyze(job)
        print(f"- {job} ({len(found)} kompetenser)")
        for s in found:
            print(f"    - {s}")


def match_jobs(jobs, analyzer, user_skills):
    """Matchar användarens kompetenser mot jobben."""
    for job in jobs:
        r = analyzer.match(job, user_skills)
        percent = f"{r['percent']}%" if r["percent"] is not None else "–"
        print(f"--- {job} ---")
        print(f"  Matchning: {percent}")
        print(f"  Matchande: {', '.join(r['matching']) or '–'}")
        print(f"  Saknade:   {', '.join(r['missing']) or '–'}")
        print()

In [8]:
analyzer = JobAnalyzer(SKILLS)

print("=== Lokala annonser ===")
local = load_local_jobs()
analyze_jobs(local, analyzer)

=== Lokala annonser ===
- AI Engineer (9 kompetenser)
    - Python
    - SQL
    - Git
    - Docker
    - AWS
    - TensorFlow
    - PyTorch
    - Machine Learning
    - NLP
- Machine Learning Engineer (7 kompetenser)
    - Python
    - Git
    - Docker
    - AWS
    - PyTorch
    - Machine Learning
    - NLP
- Data Scientist (4 kompetenser)
    - Python
    - SQL
    - TensorFlow
    - Machine Learning


In [9]:
print("=== Matchning mot lokala ===")
match_jobs(local, analyzer, ["Python", "SQL", "Git", "Docker"])

=== Matchning mot lokala ===
--- AI Engineer ---
  Matchning: 44.4%
  Matchande: docker, git, python, sql
  Saknade:   aws, machine learning, nlp, pytorch, tensorflow

--- Machine Learning Engineer ---
  Matchning: 42.9%
  Matchande: docker, git, python
  Saknade:   aws, machine learning, nlp, pytorch

--- Data Scientist ---
  Matchning: 50.0%
  Matchande: python, sql
  Saknade:   machine learning, tensorflow



In [10]:
print("=== API-jobb ===")
api_jobs = load_api_jobs()
analyze_jobs(api_jobs, analyzer)

=== API-jobb ===
- [AI] Technical Product Manager AI Stockbroking App (0 kompetenser)
- [AI] Senior Growth Product Manager AI Native (0 kompetenser)
- [AI] AI agent engineer (0 kompetenser)
- [AI] AI Trainer Image QA Evaluator (0 kompetenser)
- [AI] Technical Product Lead AI Finance App (0 kompetenser)


## Reflektion

### Vad som gick bra

- En tydlig klassstruktur med arv (`AIJob` ärver `JobAd`).
- Robust felhantering med `try/except` för fil och nätverk.
- Enkel meny med tydligt flöde.
- Användning av externa bibliotek (`requests`, `ftfy`) för att lösa verkliga problem.

### Utmaningar

- **Teckenkodning:** API:et returnerade dubbelkodad UTF-8, vilket krävde `ftfy` för att lösa.
- **HTML i beskrivningar:** Behövde rensas med regex.
- **Tomma annonser:** Vissa jobb från API:et innehöll inga av våra sökta kompetenser. Vi skiljer nu på "0% matchning" och "ingen data".

### Insikter om AI-branschen

En överraskande upptäckt var att majoriteten av AI-jobben från API:et inte kräver specifika programmeringsspråk. Detta visar att "AI-jobb" är ett brett begrepp som inkluderar produktledning, försäljning och manuell annotering – inte bara teknik.

### Vad jag skulle förbättra

- Fler kompetenser i listan (t.ex. "LLM", "Generative AI", "MLOps").
- Möjlighet att spara analysresultat till fil.
- Stöd för fler API:er som komplement till RemoteOK.